Run this notebook to visualise/analyse a signed implicit function (analytic, or from a neural sdf).
Shows the implicit values, performs marching cubes, and includes gradient visualisations.
Sliders are to change the 'slice' of the volume, and for a 'transition width' parameter.

The 'transition width' controls a regulariser that is needed for some of the neural sdfs, that can contain spurious values outside of the intended shape. When turned on, this regularisation interpolates the implicit function with a function that increases radially - but only outside the unit box, so that the sdf values inside the unit box are unaffected. The transition width controls the sharpness of this transition.

In [ ]:
# Run from the repository root, so that `import bcs` and the relative
# data/, configs/ and sdf_weights/ paths work wherever this notebook lives.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bcs').is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
from bcs.implicit_reps import *
import time
from bcs.differential import *
diffmod = DifferentialModule()

In [ ]:
device='mps'

In [ ]:
import json

# --- Choose the shape to visualise ---
# Analytic shapes need nothing else, e.g.:
#   'urchin', 'twisted_torus', 'wobbly_torus', 'rippley', 'mobius', 'analytic_torus',
#   'cylinder', 'bumpy_torus'
# DeepSDF shapes ('deep3d_*') load their checkpoint and architecture from the
# surface config(s) in configs/surfaces/ that use this sdf_id, e.g.:
#   'deep3d_bob', 'deep3d_fertility', 'deep3d_igea', 'deep3d_fandisk', 'deep3d_double_torus',
#   'deep3d_oloid', 'deep3d_dice', 'deep3d_gear', 'deep3d_cat-reference', 'deep3d_lion-reference'
shape_id = 'deep3d_bob'


def find_deepsdf_settings(sdf_id, config_dir='configs/surfaces'):
    """
    Look up how to load the DeepSDF checkpoint for `sdf_id` from the surface
    configs that use it. Returns (weights_path, model_variant, posenc).
    """
    found = set()
    for path in sorted(Path(config_dir).glob('*.json')):
        with open(path) as f:
            training_config = json.load(f).get('training-config', {})
        if training_config.get('sdf_id') == sdf_id and 'sdf_weights_path' in training_config:
            found.add((training_config['sdf_weights_path'],
                       training_config.get('model_variant'),
                       training_config.get('posenc', True)))
    if len(found) > 1:
        raise ValueError(f"Configs disagree on how to load '{sdf_id}': {sorted(found, key=str)}")
    if not found:
        print(f"No config uses '{sdf_id}'; falling back to {sdf_id}_best.ckpt "
              f"with the default architecture.")
        return f'{sdf_id}_best.ckpt', None, True
    return found.pop()


if shape_id.startswith('deep3d'):
    weights_path, model_variant, posenc = find_deepsdf_settings(shape_id)
    DEEPSDF_MODEL = load_deepsdf_model(ckpt_path='sdf_weights/surfaces/' + weights_path, device=device,
                                       model_variant=model_variant, posenc=posenc)
else:
    DEEPSDF_MODEL = None



# The implicit function used by all the plots below
implicit_func = lambda pts, transition_width:  sdf(
                pts.reshape(1, -1, 3),
                shape_id,
                squared=False,
                model=DEEPSDF_MODEL,
                transition_width=transition_width
            )


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets



def get_sdf_slice(slice_val, slice_dim='Z', resolution=20, plot_range=3.0, transition_width=0.1):
    """
    Computes SDF values for a slice along X, Y, or Z.
    """
    # 1. Create a 2D grid for the viewing plane
    # U is the horizontal axis of the plot, V is the vertical axis
    grid_coords = torch.linspace(-plot_range, plot_range, resolution, device=device)
    U, V = torch.meshgrid(grid_coords, grid_coords, indexing='ij')
    
    # 2. Map U, V to 3D coordinates (X, Y, Z) based on slice_dim
    # We create a 'Fixed' tensor filled with the slice value
    Fixed = torch.full_like(U, slice_val)
    
    if slice_dim == 'Z':
        # Plane: XY (U=X, V=Y, Fixed=Z)
        query_points = torch.stack([U, V, Fixed], dim=-1)
    elif slice_dim == 'Y':
        # Plane: XZ (U=X, V=Z, Fixed=Y)
        query_points = torch.stack([U, Fixed, V], dim=-1)
    elif slice_dim == 'X':
        # Plane: YZ (U=Y, V=Z, Fixed=X)
        query_points = torch.stack([Fixed, U, V], dim=-1)
    else:
        raise ValueError("Invalid slice dimension")

    # Reshape for batch inference (1, N, 3)
    query_points = query_points.reshape(1, -1, 3)
    
    # 3. Run Inference
    with torch.no_grad():
        result = implicit_func(query_points, transition_width=transition_width)
        
    if device == 'mps': torch.mps.synchronize()
    elif device == 'cuda': torch.cuda.synchronize()
        
    # Reshape back to image format (Resolution, Resolution)
    return result.squeeze().cpu().numpy().reshape(resolution, resolution)

def interactive_plot(slice_pos=0.0, slice_dim='Z', transition_width=0.1):
    """
    Update function for the widget.
    """
    resolution = 300
    limit = 0.4
    plot_range = 2.0
    
    # Get data
    sdf_values = get_sdf_slice(slice_pos, slice_dim, resolution, plot_range, transition_width)
    
    plt.figure(figsize=(7, 6))
    extent_bounds = (-plot_range, plot_range, -plot_range, plot_range)
    
    # 1. Heatmap
    im = plt.imshow(
        sdf_values,
        origin='lower',
        cmap='RdBu',
        vmin=-limit,
        vmax=limit,
        extent=extent_bounds
    )
    
    # 2. Zero Level Set (Thick Line)
    plt.contour(
        sdf_values,
        levels=[0],
        colors='black',
        linewidths=2,
        origin='lower',
        extent=extent_bounds
    )

    # 3. Background Contour Grid (Thin Lines)
    # Reduced range/density slightly for performance, adjust as needed
    plt.contour(
        sdf_values,
        levels=np.arange(-5, 5, 0.1), 
        colors='black',
        linewidths=0.3, # Made thinner for readability
        alpha=0.5,
        origin='lower',
        extent=extent_bounds
    )
    
    plt.colorbar(im, label='SDF Value')
    plt.title(f'Slice along {slice_dim}-axis at {slice_pos:.2f}')
    
    # Dynamic Axis Labels
    if slice_dim == 'Z':
        plt.xlabel('X axis')
        plt.ylabel('Y axis')
    elif slice_dim == 'Y':
        plt.xlabel('X axis')
        plt.ylabel('Z axis')
    elif slice_dim == 'X':
        plt.xlabel('Y axis')
        plt.ylabel('Z axis')

    plt.show()

# --- 3. Define Widgets ---

# Dropdown for selecting the Axis
axis_dropdown = Dropdown(
    options=['X', 'Y', 'Z'],
    value='Z',
    description='Axis:',
)

# Slider for the position along that axis
pos_slider = FloatSlider(
    value=0.0,
    min=-2.0,
    max=2.0,
    step=0.05,
    description='Slice Pos:',
    continuous_update=True # Set False if it lags
)


transition_width_slider = FloatSlider(
    value=0.6,
    min=0.0,
    max=10.0,
    step=0.1,
    description='Transition Width:',
    continuous_update=True
)

# --- 4. Launch Interface ---
interact(
    interactive_plot, 
    slice_pos=pos_slider, 
    slice_dim=axis_dropdown,
    transition_width=transition_width_slider
);

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

def get_sdf_gradmag_slice(
    slice_val,
    slice_dim='Z',
    resolution=100,
    plot_range=3.0,
    transition_width=0.1
):
    """
    Computes ||∇SDF|| via autograd for a slice along X, Y, or Z.
    """
    # 1. Create grid
    grid_coords = torch.linspace(-plot_range, plot_range, resolution, device=device)
    U, V = torch.meshgrid(grid_coords, grid_coords, indexing='ij')

    Fixed = torch.full_like(U, slice_val)

    if slice_dim == 'Z':
        query_points = torch.stack([U, V, Fixed], dim=-1)
    elif slice_dim == 'Y':
        query_points = torch.stack([U, Fixed, V], dim=-1)
    elif slice_dim == 'X':
        query_points = torch.stack([Fixed, U, V], dim=-1)
    else:
        raise ValueError("Invalid slice dimension")

    query_points = query_points.reshape(1, -1, 3)
    query_points.requires_grad_(True)

    # 2. Forward pass (NO no_grad!)
    sdf_vals = implicit_func(query_points, transition_width=transition_width)

    # 3. Compute gradient magnitude
    grad = torch.autograd.grad(
        outputs=sdf_vals.sum(),
        inputs=query_points,
        create_graph=False,
        retain_graph=False
    )[0]

    grad_mag = grad.norm(dim=-1)

    if device == 'mps':
        torch.mps.synchronize()
    elif device == 'cuda':
        torch.cuda.synchronize()

    return grad_mag.squeeze().detach().cpu().numpy().reshape(resolution, resolution)


def interactive_gradmag_plot(
    slice_pos=0.0,
    slice_dim='Z',
    transition_width=0.1
):
    resolution = 100
    plot_range = 1.0

    gradmag = get_sdf_gradmag_slice(
        slice_pos,
        slice_dim,
        resolution,
        plot_range,
        transition_width
    )

    plt.figure(figsize=(7, 6))
    extent_bounds = (-plot_range, plot_range, -plot_range, plot_range)

    # Fixed color scale: 0 (black) = zero gradient
    im = plt.imshow(
        gradmag,
        origin='lower',
        cmap='viridis',
        vmin=0.0,        # zero gradient always same color
        vmax=5.0,        # raise if gradients exceed this
        extent=extent_bounds
    )

    plt.colorbar(im, label='||∇SDF||')
    plt.title(f'||∇SDF|| slice along {slice_dim}-axis at {slice_pos:.2f}')

    if slice_dim == 'Z':
        plt.xlabel('X axis')
        plt.ylabel('Y axis')
    elif slice_dim == 'Y':
        plt.xlabel('X axis')
        plt.ylabel('Z axis')
    elif slice_dim == 'X':
        plt.xlabel('Y axis')
        plt.ylabel('Z axis')

    plt.show()


# --- Widgets ---
axis_dropdown = Dropdown(
    options=['X', 'Y', 'Z'],
    value='Z',
    description='Axis:',
)

pos_slider = FloatSlider(
    value=0.0,
    min=-2.0,
    max=2.0,
    step=0.05,
    description='Slice Pos:',
    continuous_update=True
)

transition_width_slider = FloatSlider(
    value=0.6,
    min=0.0,
    max=100.0,
    step=0.1,
    description='Transition Width:',
    continuous_update=True
)

interact(
    interactive_gradmag_plot,
    slice_pos=pos_slider,
    slice_dim=axis_dropdown,
    transition_width=transition_width_slider
);


In [ ]:
def interactive_gradmag_plot(
    slice_pos=0.0,
    slice_dim='Z',
    transition_width=0.1,
    highlight_thresh=100.0,   # <---- threshold for “bad” gradients
):
    resolution = 100
    plot_range = 2.0

    gradmag = get_sdf_gradmag_slice(
        slice_pos,
        slice_dim,
        resolution,
        plot_range,
        transition_width
    )

    plt.figure(figsize=(7, 6))
    extent_bounds = (-plot_range, plot_range, -plot_range, plot_range)

    # --- Base plot (normal gradients) ---
    im = plt.imshow(
        gradmag,
        origin='lower',
        cmap='viridis',
        vmin=0.0,
        vmax=1.0,        # keep interpretable for valid SDF regions
        extent=extent_bounds
    )

    # --- Highlight pathological gradients ---
    bad_mask = gradmag > highlight_thresh

    print(gradmag.min(), gradmag.max())

    if np.any(bad_mask):
        # Red overlay where gradients explode
        plt.imshow(
            bad_mask,
            origin='lower',
            cmap='Reds',
            alpha=0.9,      # strong highlight
            extent=extent_bounds
        )

    plt.colorbar(im, label='||∇SDF|| (clipped at 1.0)')
    plt.title(
        f'||∇SDF|| slice along {slice_dim}-axis at {slice_pos:.2f}\n'
        f'Red = ||∇SDF|| > {highlight_thresh:g}'
    )

    if slice_dim == 'Z':
        plt.xlabel('X axis')
        plt.ylabel('Y axis')
    elif slice_dim == 'Y':
        plt.xlabel('X axis')
        plt.ylabel('Z axis')
    elif slice_dim == 'X':
        plt.xlabel('Y axis')
        plt.ylabel('Z axis')

    plt.show()


interact(
    interactive_gradmag_plot,
    slice_pos=pos_slider,
    slice_dim=axis_dropdown,
    transition_width=transition_width_slider
);

In [ ]:
import torch
import numpy as np
from skimage import measure
import trimesh

def make_3d_grid(min_val, max_val, resolution, device):
    coords = torch.linspace(min_val, max_val, resolution, device=device)
    X, Y, Z = torch.meshgrid(coords, coords, coords, indexing='ij')
    return torch.stack([X, Y, Z], dim=-1).reshape(-1, 3)

# --- Configuration ---
res = 64
min_val = -1.1
max_val = 1.1
device = 'mps'

# 1. Generate Points
points_3d = make_3d_grid(min_val, max_val, res, device)

# 2. Inference
with torch.no_grad():
    sdf_flat = implicit_func(
        points_3d,
        transition_width=10.0
    ).cpu().numpy()

sdf_volume = sdf_flat.reshape(res, res, res)

# 3. Marching Cubes WITH SPACING
voxel_size = (max_val - min_val) / (res - 1)

verts, faces, normals, values = measure.marching_cubes(
    sdf_volume,
    level=0.0,
    spacing=(voxel_size, voxel_size, voxel_size)
)



print('num_faces', faces.shape[0])


# 4. Shift origin to min_val
# marching_cubes assumes origin at (0,0,0)
verts += np.array([min_val, min_val, min_val])

# 5. Export
mesh = trimesh.Trimesh(
    vertices=verts,
    faces=faces,
    vertex_normals=normals*-1
)
mesh.export(f'data/surfaces/{shape_id}-mc.ply')
mesh.show()





import open3d as o3d

# 1. Convert the trimesh mesh to Open3D
mesh_o3d = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(mesh.vertices),
    triangles=o3d.utility.Vector3iVector(mesh.faces),
)

# Use the marching-cubes normals (sign flipped)
mesh_o3d.vertex_normals = o3d.utility.Vector3dVector(normals * -1)

# 2. Set a pale blue color for all vertices
pale_blue = np.array([0.6, 0.8, 1.0])  # RGB in [0,1]
mesh_o3d.vertex_colors = o3d.utility.Vector3dVector(
    np.tile(pale_blue, (len(verts), 1))
)


mesh_o3d.compute_vertex_normals()
# 3. Show in Open3D viewer
o3d.visualization.draw_geometries([mesh_o3d])

In [ ]:
import torch

def max_gradmag_in_block(
    resolution=200,
    plot_range=2.0,
    transition_width=0.1,
):
    """
    Computes max ||∇SDF|| over a cubic block [-plot_range, plot_range]^3
    """

    # 1. Create 3D grid
    coords = torch.linspace(-plot_range, plot_range, resolution, device=device)
    X, Y, Z = torch.meshgrid(coords, coords, coords, indexing='ij')

    query_points = torch.stack([X, Y, Z], dim=-1)   # [R, R, R, 3]
    query_points = query_points.reshape(1, -1, 3)
    query_points.requires_grad_(True)

    # 2. Forward pass
    sdf_vals = implicit_func(query_points, transition_width=transition_width)

    # 3. Compute gradient
    grad = torch.autograd.grad(
        outputs=sdf_vals.sum(),
        inputs=query_points,
        create_graph=False,
        retain_graph=False
    )[0]

    grad_mag = grad.norm(dim=-1)

    if device == 'mps':
        torch.mps.synchronize()
    elif device == 'cuda':
        torch.cuda.synchronize()

    max_val = grad_mag.max().item()
    mean_val = grad_mag.mean().item()

    print(f"Grid resolution: {resolution}³")
    print(f"||∇SDF|| mean: {mean_val:.6f}")
    print(f"||∇SDF|| max : {max_val:.6f}")

    return max_val


max_gradmag_in_block(
    resolution=100,
    plot_range=2.0,
    transition_width=0.6
)

